# Load Pfam term definitions → `nmdc_ref_data.pfam_terms` (issue #100)

Downloads `Pfam-A.clans.tsv.gz` from the EBI/InterPro FTP site (494 KB compressed),
parses it into a reference table, and registers it as a Delta table in the new
`nmdc_ref_data` namespace.

| Column | Type | Description |
|---|---|---|
| `pfam_id` | string | Pfam accession without version (e.g. `PF00001`) — join key to GFF tables |
| `name` | string | Short name (e.g. `7tm_1`) |
| `description` | string | Human-readable description |
| `clan_id` | string (nullable) | Clan accession (e.g. `CL0192`); null if family has no clan |
| `clan_name` | string (nullable) | Clan short name (e.g. `GPCR_A`); null if no clan |

Source: `https://ftp.ebi.ac.uk/pub/databases/Pfam/current_release/Pfam-A.clans.tsv.gz`  
License: CC0 (Pfam → InterPro, openly redistributable)

Refresh: re-run this notebook after each Pfam release to pick up new/updated families.

### Configuration

In [1]:
import gzip
import io
import json
from pathlib import Path

import pandas as pd
import requests

TENANT  = "nmdc"
DATASET = "ref_data"
BUCKET  = "cdm-lake"

PFAM_CLANS_URL = "https://ftp.ebi.ac.uk/pub/databases/Pfam/current_release/Pfam-A.clans.tsv.gz"
TABLE_NAME     = "pfam_terms"
OUT_DIR        = Path("loaded_pfam_terms")
OUT_DIR.mkdir(exist_ok=True)

NAMESPACE     = f"{TENANT}_{DATASET}"
BRONZE_PREFIX = f"tenant-general-warehouse/{TENANT}/datasets/{DATASET}"
BRONZE_BASE   = f"s3a://{BUCKET}/{BRONZE_PREFIX}"
SILVER_BASE   = f"s3a://{BUCKET}/tenant-sql-warehouse/{TENANT}/{NAMESPACE}.db"

print(f"Tenant         : {TENANT}")
print(f"Namespace      : {NAMESPACE}")
print(f"Bronze base    : {BRONZE_BASE}")
print(f"Silver base    : {SILVER_BASE}")
print(f"Table          : {NAMESPACE}.{TABLE_NAME}")
print(f"Source URL     : {PFAM_CLANS_URL}")

Tenant         : nmdc
Namespace      : nmdc_ref_data
Bronze base    : s3a://cdm-lake/tenant-general-warehouse/nmdc/datasets/ref_data
Silver base    : s3a://cdm-lake/tenant-sql-warehouse/nmdc/nmdc_ref_data.db
Table          : nmdc_ref_data.pfam_terms
Source URL     : https://ftp.ebi.ac.uk/pub/databases/Pfam/current_release/Pfam-A.clans.tsv.gz


### Spark + MinIO setup

In [ ]:
from berdl_notebook_utils.spark.connect_server import (
    start_spark_connect_server,
    SparkConnectServerConfig,
)

# Force-restart so the server picks up current group memberships.
# compute_allowed_namespace_prefixes() shows what SHOULD be allowed, but the
# running server was launched with whatever prefixes it had at start time.
# nmdc_ref_data creation fails if the server pre-dates the nmdc tenant membership.
print("Restarting Spark Connect server to refresh allowed namespace prefixes...")
start_spark_connect_server(force_restart=True)

_prefixes = SparkConnectServerConfig().compute_allowed_namespace_prefixes()
print(f"Server restarted. Active prefixes: {_prefixes}")
assert "nmdc_" in _prefixes.split(","), f"nmdc_ missing after restart — check nmdc group membership"

In [3]:
spark = get_spark_session(app_name=f"load_{NAMESPACE}_{TABLE_NAME}", tenant_name=TENANT)
mc    = get_minio_client()
print(f"Spark version: {spark.version}")
try:
    print(f"MinIO endpoint: {mc._base_url._url.netloc}")
except AttributeError:
    print("MinIO endpoint: (not directly inspectable)")

Spark version: 4.0.1
MinIO endpoint: minio.berdl.kbase.us


### Download and parse Pfam-A.clans.tsv.gz

Tab-separated, no header, 5 columns:
`pfam_id \t clan_id \t clan_name \t name \t description`

Families without a clan assignment have empty `clan_id` and `clan_name`.

In [4]:
print(f"Downloading {PFAM_CLANS_URL} ...")
resp = requests.get(PFAM_CLANS_URL, timeout=60)
resp.raise_for_status()
print(f"Downloaded {len(resp.content) / 1024:.0f} KB")

with gzip.open(io.BytesIO(resp.content)) as f:
    raw = f.read().decode("utf-8")

rows = []
for line in raw.splitlines():
    if not line.strip():
        continue
    parts = line.split("\t")
    if len(parts) != 5:
        continue
    pfam_id, clan_id, clan_name, name, description = parts
    rows.append({
        "pfam_id":     pfam_id.strip(),
        "name":        name.strip(),
        "description": description.strip(),
        "clan_id":     clan_id.strip() or None,
        "clan_name":   clan_name.strip() or None,
    })

df = pd.DataFrame(rows)
print(f"Parsed {len(df):,} entries")
print(f"  With clan:    {df['clan_id'].notna().sum():,}")
print(f"  Without clan: {df['clan_id'].isna().sum():,}")
print(f"  Null name:    {df['name'].isna().sum()}")
print(f"  Null desc:    {df['description'].isna().sum()}")
print()
df.head()

Downloaded 494 KB
Parsed 27,481 entries
  With clan:    12,486
  Without clan: 14,995
  Null name:    0
  Null desc:    0



,pfam_id,name,description,clan_id,clan_name
0,PF00001,7tm_1,7 transmembrane receptor (rhodopsin family),CL0192,GPCR_A
1,PF00002,7tm_2,7 transmembrane receptor (Secretin family),CL0192,GPCR_A
2,PF00003,7tm_3,7 transmembrane sweet-taste receptor of 3 GCPR,CL0192,GPCR_A
3,PF00004,AAA,ATPase family associated with various cellular...,CL0023,P-loop_NTPase
4,PF00005,ABC_tran,ABC transporter,CL0023,P-loop_NTPase


In [5]:
local_parquet = OUT_DIR / f"{TABLE_NAME}.parquet"
df.to_parquet(local_parquet, index=False)
print(f"Written {local_parquet} ({local_parquet.stat().st_size / 1024:.0f} KB)")

Written loaded_pfam_terms/pfam_terms.parquet (958 KB)


### Upload to Bronze

In [6]:
object_key = f"{BRONZE_PREFIX}/{TABLE_NAME}.parquet"
bronze_uri = f"s3a://{BUCKET}/{object_key}"

print(f"Uploading → {bronze_uri}")
mc.fput_object(BUCKET, object_key, str(local_parquet))
print("Upload complete.")

Uploading → s3a://cdm-lake/tenant-general-warehouse/nmdc/datasets/ref_data/pfam_terms.parquet
Upload complete.


### Ingest to Silver (`nmdc_ref_data.pfam_terms`)

In [7]:
config = {
    "tenant":  TENANT,
    "dataset": DATASET,
    "paths": {
        "bronze_base": BRONZE_BASE,
        "silver_base": SILVER_BASE,
    },
    "defaults": {
        "parquet": {},
    },
    "tables": [
        {
            "name":        TABLE_NAME,
            "format":      "parquet",
            "bronze_path": f"{TABLE_NAME}.parquet",
            "write_mode":  "overwrite",
        }
    ],
}

print(json.dumps(config, indent=2))

{
  "tenant": "nmdc",
  "dataset": "ref_data",
  "paths": {
    "bronze_base": "s3a://cdm-lake/tenant-general-warehouse/nmdc/datasets/ref_data",
    "silver_base": "s3a://cdm-lake/tenant-sql-warehouse/nmdc/nmdc_ref_data.db"
  },
  "defaults": {
    "parquet": {}
  },
  "tables": [
    {
      "name": "pfam_terms",
      "format": "parquet",
      "bronze_path": "pfam_terms.parquet",
      "write_mode": "overwrite"
    }
  ]
}


In [8]:
report = ingest(config=config, spark=spark, minio_client=mc)
print(json.dumps(report, indent=2, default=str))

{"time": "2026-05-01 18:42:38,385", "pipeline": "data_lakehouse_ingest", "schema": "default", "table": "pipeline_stage", "level": "INFO", "module": "config_loader", "msg": "Using inline dict configuration"}
{"time": "2026-05-01 18:42:38,385", "pipeline": "data_lakehouse_ingest", "schema": "default", "table": "pipeline_stage", "level": "INFO", "module": "config_loader", "msg": "Config loaded successfully"}
{"time": "2026-05-01 18:42:38,386", "pipeline": "data_lakehouse_ingest", "schema": "default", "table": "pipeline_stage", "level": "INFO", "module": "config_loader", "msg": "Table 'pfam_terms' has no explicit schema ('schema_sql'/'schema'); schema will be inferred by the loader."}
{"time": "2026-05-01 18:42:38,386", "pipeline": "data_lakehouse_ingest", "schema": "default", "table": "pipeline_stage", "level": "INFO", "module": "config_loader", "msg": "Minimal config validation passed"}
{"time": "2026-05-01 18:42:38,386", "pipeline": "data_lakehouse_ingest", "schema": "default", "table":

SparkConnectGrpcException: <_MultiThreadedRendezvous of RPC that terminated with:
	status = StatusCode.PERMISSION_DENIED
	details = "Database name 'nmdc_ref_data' is not allowed. Database names must start with one of the following prefixes: [u_mamillerpa__]. Use create_namespace_if_not_exists() to create databases with the correct prefix."
	debug_error_string = "UNKNOWN:Error received from peer  {grpc_status:7, grpc_message:"Database name \'nmdc_ref_data\' is not allowed. Database names must start with one of the following prefixes: [u_mamillerpa__]. Use create_namespace_if_not_exists() to create databases with the correct prefix."}"
>

### Smoke test

In [ ]:
result = spark.sql(f"""
    SELECT pfam_id, name, description, clan_id, clan_name
    FROM {NAMESPACE}.{TABLE_NAME}
    ORDER BY pfam_id
    LIMIT 10
""").toPandas()

print(f"Row count: {spark.sql(f'SELECT COUNT(*) FROM {NAMESPACE}.{TABLE_NAME}').collect()[0][0]:,}")
display(result)

In [ ]:
# Verify the motivating demo domains are present
demo = spark.sql(f"""
    SELECT pfam_id, name, description
    FROM {NAMESPACE}.{TABLE_NAME}
    WHERE pfam_id IN ('PF04183', 'PF06276')
""").toPandas()

print("Siderophore/iron-reductase demo domains:")
display(demo)